# Calendar Dimension Table

Builds `man_catlog.caresync_schema_gold.calendar`, a date dimension with one row per day from 1940-01-01 to 2030-12-31 and standard date attributes (year, quarter, month, week, day, flags, period boundaries).

In [0]:
%sql
CREATE OR REPLACE TABLE man_catlog.caresync_schema_gold.calendar
COMMENT 'Calendar date dimension, one row per day from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS cal_date
)
SELECT
  CAST(date_format(cal_date, 'yyyyMMdd') AS INT)              AS date_key,
  cal_date                                                   AS date,
  year(cal_date)                                             AS year,
  CASE WHEN month(cal_date) <= 6 THEN 1 ELSE 2 END           AS half_year,
  quarter(cal_date)                                          AS quarter,
  concat('Q', quarter(cal_date))                             AS quarter_name,
  concat(year(cal_date), '-Q', quarter(cal_date))            AS year_quarter,
  month(cal_date)                                            AS month,
  date_format(cal_date, 'MMMM')                              AS month_name,
  date_format(cal_date, 'MMM')                               AS month_short_name,
  date_format(cal_date, 'yyyy-MM')                           AS year_month,
  CAST(date_format(cal_date, 'yyyyMM') AS INT)               AS year_month_key,
  weekofyear(cal_date)                                       AS week_of_year,        -- ISO 8601 week
  extract(YEAROFWEEK FROM cal_date)                          AS iso_week_year,
  CAST(ceil((dayofmonth(cal_date) + extract(DOW_ISO FROM trunc(cal_date, 'MM')) - 1) / 7.0) AS INT) AS week_of_month, -- Monday-start weeks
  CAST(date_trunc('WEEK', cal_date) AS DATE)                 AS week_start_date,     -- Monday
  date_add(CAST(date_trunc('WEEK', cal_date) AS DATE), 6)    AS week_end_date,       -- Sunday
  dayofmonth(cal_date)                                       AS day_of_month,
  dayofyear(cal_date)                                        AS day_of_year,
  extract(DOW_ISO FROM cal_date)                             AS day_of_week,         -- 1=Monday ... 7=Sunday
  date_format(cal_date, 'EEEE')                              AS day_name,
  date_format(cal_date, 'EEE')                               AS day_short_name,
  CASE WHEN extract(DOW_ISO FROM cal_date) IN (6, 7) THEN true ELSE false END AS is_weekend,
  CASE WHEN extract(DOW_ISO FROM cal_date) IN (6, 7) THEN false ELSE true END AS is_weekday,
  trunc(cal_date, 'MM')                                      AS first_day_of_month,
  last_day(cal_date)                                         AS last_day_of_month,
  dayofmonth(last_day(cal_date))                             AS days_in_month,
  cal_date = last_day(cal_date)                              AS is_month_end,
  trunc(cal_date, 'QUARTER')                                 AS first_day_of_quarter,
  last_day(add_months(trunc(cal_date, 'QUARTER'), 2))        AS last_day_of_quarter,
  datediff(cal_date, trunc(cal_date, 'QUARTER')) + 1         AS day_of_quarter,
  trunc(cal_date, 'YEAR')                                    AS first_day_of_year,
  make_date(year(cal_date), 12, 31)                          AS last_day_of_year,
  (year(cal_date) % 4 = 0 AND year(cal_date) % 100 <> 0) OR year(cal_date) % 400 = 0 AS is_leap_year
FROM dates
ORDER BY cal_date

In [0]:
%sql
SELECT count(*) AS row_count,
       count(DISTINCT date) AS distinct_dates,
       min(date) AS min_date,
       max(date) AS max_date,
       datediff(max(date), min(date)) + 1 AS expected_rows
FROM man_catlog.caresync_schema_gold.calendar

In [0]:
%sql
select * FROM man_catlog.caresync_schema_gold.calendar